In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv('cardekho_imputated.csv',index_col="Unnamed: 0")
df.head()

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15411 entries, 0 to 19543
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   car_name           15411 non-null  object 
 1   brand              15411 non-null  object 
 2   model              15411 non-null  object 
 3   vehicle_age        15411 non-null  int64  
 4   km_driven          15411 non-null  int64  
 5   seller_type        15411 non-null  object 
 6   fuel_type          15411 non-null  object 
 7   transmission_type  15411 non-null  object 
 8   mileage            15411 non-null  float64
 9   engine             15411 non-null  int64  
 10  max_power          15411 non-null  float64
 11  seats              15411 non-null  int64  
 12  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(5), object(6)
memory usage: 1.6+ MB


In [4]:
df.drop(columns=['car_name'],axis=1,inplace=True)

In [5]:
X=df.drop(['selling_price'],axis=1)
y=df['selling_price']

In [6]:
cat_features=X.select_dtypes(include='object').columns
numeric_features=X.select_dtypes(exclude='object').columns

In [7]:
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
numeric_transformer=StandardScaler()
cat_transformer=OneHotEncoder(
    min_frequency=5,
    handle_unknown='infrequent_if_exist'
)

preprocessor=ColumnTransformer(
    [
        ('oh_encoder',cat_transformer,cat_features),
        ('Standard Scaler',numeric_transformer,numeric_features)
    ]
)

In [8]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=10)

In [9]:
X_train=preprocessor.fit_transform(X_train)
X_test=preprocessor.transform(X_test)

In [10]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,root_mean_squared_error,r2_score

In [11]:
y_train

1564      390000
18554     175000
18483    2950000
13796     750000
4460      428000
          ...   
14770     775000
1726     2700000
16247     285000
9326      225000
1657      590000
Name: selling_price, Length: 12328, dtype: int64

In [13]:
models={
    'Random Forst':RandomForestRegressor(),
    'DecisionTreeRegressor':DecisionTreeRegressor()
}

for i in range(len(list(models))):
    model=list(models.values())[i]

    #train model
    model.fit(X_train,y_train)

    # Make Predictions
    y_pred_train=model.predict(X_train)
    y_pred_test=model.predict(X_test)

    # Training data performance
    train_mae=mean_absolute_error(y_train,y_pred_train)
    train_mse=mean_squared_error(y_train,y_pred_train)
    train_rmse=root_mean_squared_error(y_train,y_pred_train)
    train_r2=r2_score(y_train,y_pred_train)

    # Testing data performance
    test_mae=mean_absolute_error(y_test,y_pred_test)
    test_mse=mean_squared_error(y_test,y_pred_test)
    test_rmse=root_mean_squared_error(y_test,y_pred_test)
    test_r2=r2_score(y_test,y_pred_test)

    print(list(models.keys())[i])
    print('Training Data performance')
    print('-Train MAE:{:.4f}'.format(train_mae))
    print('-Train MSE:{:.4f}'.format(train_mse))
    print('-Train RMSE:{:.4f}'.format(train_rmse))
    print('-Train R2:{:.4f}'.format(train_r2))

    print('\n','-'*30)

    print('Test Data performance')
    print('-Test MAE:{:.4f}'.format(test_mae))
    print('-Test MSE:{:.4f}'.format(test_mse))
    print('-Test RMSE:{:.4f}'.format(test_rmse))
    print('-Test R2:{:.4f}'.format(test_r2))
    print('\n','='*35,'\n')

Random Forst
Training Data performance
-Train MAE:37590.8232
-Train MSE:7601131487.8914
-Train RMSE:87184.4682
-Train R2:0.9893

 ------------------------------
Test Data performance
-Test MAE:106771.3382
-Test MSE:273192289625.4958
-Test RMSE:522677.9980
-Test R2:0.7653


DecisionTreeRegressor
Training Data performance
-Train MAE:4814.6279
-Train MSE:383900402.8769
-Train RMSE:19593.3765
-Train R2:0.9995

 ------------------------------
Test Data performance
-Test MAE:129371.9105
-Test MSE:157293502366.2018
-Test RMSE:396602.4488
-Test R2:0.8648




In [14]:
rf_params={
    'n_estimators':[100,10,40,400],
    'max_features':[6,4,3,None,'sqrt'],
    'max_depth':[3,6,8,None]
}

In [18]:
RandomCv=[
    ('RF',RandomForestRegressor(),rf_params)
]

In [19]:
from sklearn.model_selection import RandomizedSearchCV

model_param={}
for name,model,params in RandomCv:
    random=RandomizedSearchCV(estimator=model,param_distributions=params,n_iter=100,n_jobs=-1,verbose=2,cv=3)
    random.fit(X_train,y_train)
    model_param[name]=random.best_params_

for model_name in model_param:
    print('*'*10,'Best Params for model {}'.format(model_name),'*'*10,)
    print(model_param[model_name])

C:\Users\ABRAR\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 80 is smaller than n_iter=100. Running 80 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 80 candidates, totalling 240 fits
********** Best Params for model RF **********
{'n_estimators': 400, 'max_features': None, 'max_depth': None}


In [20]:
models={
    'RandomForest':RandomForestRegressor(n_estimators=400,max_features=None,max_depth=None)
}

for i in range(len(list(models))):
    model=list(models.values())[i]

    #train model
    model.fit(X_train,y_train)

    # Make Predictions
    y_pred_train=model.predict(X_train)
    y_pred_test=model.predict(X_test)

    # Training data performance
    train_mae=mean_absolute_error(y_train,y_pred_train)
    train_mse=mean_squared_error(y_train,y_pred_train)
    train_rmse=root_mean_squared_error(y_train,y_pred_train)
    train_r2=r2_score(y_train,y_pred_train)

    # Testing data performance
    test_mae=mean_absolute_error(y_test,y_pred_test)
    test_mse=mean_squared_error(y_test,y_pred_test)
    test_rmse=root_mean_squared_error(y_test,y_pred_test)
    test_r2=r2_score(y_test,y_pred_test)

    print(list(models.keys())[i])
    print('Training Data performance')
    print('-Train MAE:{:.4f}'.format(train_mae))
    print('-Train MSE:{:.4f}'.format(train_mse))
    print('-Train RMSE:{:.4f}'.format(train_rmse))
    print('-Train R2:{:.4f}'.format(train_r2))

    print('\n','-'*30)

    print('Test Data performance')
    print('-Test MAE:{:.4f}'.format(test_mae))
    print('-Test MSE:{:.4f}'.format(test_mse))
    print('-Test RMSE:{:.4f}'.format(test_rmse))
    print('-Test R2:{:.4f}'.format(test_r2))
    print('\n','='*35,'\n')

RandomForest
Training Data performance
-Train MAE:37398.6095
-Train MSE:8039837087.5462
-Train RMSE:89665.1386
-Train R2:0.9886

 ------------------------------
Test Data performance
-Test MAE:106137.1595
-Test MSE:267836379482.3509
-Test RMSE:517529.1098
-Test R2:0.7699


